## Situación de Knapsack 0/1

### Enunciado

El grupo *Raccon* es un equipo de arquitectura de software dedicado a desarrollar el sistema de software *Raccon Analytics*, un analizador de tendencias de redes sociales en el cuál el usuario final puede hacer búsquedas tentativas para youtube o google trends, y el sistema devuelve búsquedas relacionadas y keywords inferidas con inteligencia artificial, junto con contenido reciente y viral de dichas redes sociales, para que el usuario tenga información valiosa para poder subir contenido público en las redes y viralizarlo.

Este grupo actualmente tiene sus componentes del sistema desplegados de manera distribuida en 4 máquinas de cómputo diferentes; cómo este grupo es muy profesional y se preocupa por la calidad del software, están tomando decisiones arquitectónicas para que su sistema cumpla con el atributo de calidad de confiabilidad, que consiste en garantizar una buena disponibilidad, resiliencia y tolerancia a fallos.
El mes que viene (Julio de 2026) comienza uno de los eventos más grandes de internet organizado por uno de los creadores de contenido más grandes, se aproxima la velada del año en su sexta edición organizada por Ibai Llanos, y con sigo muchos creadores más pequeños verán oportuno obtener visitas gracias a ello, y usarán la aplicación *Raccon Analytics* para evaluar tendencias y subir el contenido correcto.

Actualmente el sistema desplegado en las 4 computadoras soporta un estimado de 300 usuarios concurrentes, pero debido al incremento de usuarios que generará la velada, el equipo *Raccon* deberá garantizar un correcto funcionamiento con 1000 usuarios concurrentes, por lo que optan por contratar temporalmente poder computacional en los servicios de AWS, y desplegar allí réplicas de los componentes de mayor impacto en el negocio junto con balanceadores de carga y un service discovery.

El poder de cómputo a nivel de procesamiento que ofrece el contrato es más que suficiente para el equipo *Raccon* ya que el sistema mayormente solo maneja el flujo de datos y peticiones junto con API's externas, pero donde sí hay un limitante importante es en la memoria RAM.
Dicho contrato ofrece un cluster con un total de 8192MB de RAM, y dado que el equipo desplegará docker swarm junto con sus balanceadores y su DNS (Service Discovery), la memoria que queda para componentes del sistema se software son 8000MB.

Está claro que desplegar la totalidad de componentes del sistema en AWS no es viable, ya que con 1000 usuarios concurrentes, simplemente la RAM no daría abasto, por lo que por medio de pruebas de rendimiento desarrolladas por el equipo, por cada componente individual se obtuvo la cantidad de RAM en megabytes (MB) necesaria para ejecutarlo, y los resultados los anotaron en un listado de pesos denominado $W$.
Adicionalmente, hubo una discusión por parte del equipo sobre el nivel de impacto en el negocio que generaba cada uno de los componentes del sistema, y a cada uno le asignaron un número entero positivo de manera tal que un número alto indicaba alto nivel de impacto e importancia, mientras que un número bajo indicaba bajo nivel de impacto e importancia, y estos valores de impacto los anotaron en un listado de valores denominado $V$.

El equipo quiere maximizar la sumatoria de niveles de impacto de los componentes desplegados en AWS, y los componentes que no se desplieguen en AWS se mantendrán desplegados en las máquinas del equipo.

### Modelamiento matemático del problema

Definimos las listas de pesos y de valores:

$W = \{601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110\}$

$V = \{100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930\}$

$|W| = |V| = 50$

$W_i$ es el costo en RAM del i-ésimo item

$V_i$ es el impacto estimado del i-ésimo item

#### Variables de decisión

Hay 50 variables binarias, $X_i$ es una variable binaria que indica si se elije o no el i-ésimo item

$0 \leq i < |V|$

#### Función objetivo

$Max$ $z =$ $\sum_{i=0}^{49}$ $V_i$ $\cdot$ $X_i$ unidades de impacto

#### Restricciones

Restricción binaria: $X_i \in \{0, 1\}$ $\forall$ $i \in \mathbb{Z} \cap [0, 49]$

Restricción de máximo uso de RAM: $\sum_{i=0}^{49}$ $W_i$ $\cdot$ $X_i$ $\leq$ $8000$

#### Solución con programación dinámica

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cassert>
#include <cstring>
using namespace std;

In [2]:
const int mxW = 8000;

In [3]:
vector<bool> Knapsack(vector<int>& weig, vector<int>& val) {
  int N = weig.size();
  int dp[N][mxW + 1];
  bool take[N][mxW + 1];
  memset(dp, 0, sizeof(dp));
  memset(take, 0, sizeof(take));

  dp[0][weig[0]] = val[0];
  take[0][weig[0]] = true;

  for (int phase = 1; phase < N; ++phase) {
    int idx = phase;
    for (int w = 0; w <= mxW; ++w) {
      // Dont take
      dp[idx][w] = dp[idx - 1][w];

      // Take
      if (weig[idx] <= w && dp[idx - 1][w - weig[idx]] + val[idx] >= dp[idx][w]) {
        dp[idx][w] = dp[idx - 1][w - weig[idx]] + val[idx];
        take[idx][w] = true;
      }
    }
  }
  int best_val = *max_element(dp[N - 1], dp[N - 1] + mxW + 1);

  // Build the solution
  int current_weig = -1;
  for (int w = 0; w <= mxW; ++w) {
    if (best_val == dp[N - 1][w]) {
      current_weig = w;
      break;
    }
  }
  assert(current_weig > -1);
  int check_val = 0;
  vector<bool> X(N, false);
  for (int i = N - 1; i >= 0; --i) {
    if (take[i][current_weig]) {
      X[i] = true;
      current_weig -= weig[i];
      check_val += val[i];
    }
  }
  assert(current_weig == 0);
  assert(best_val == check_val);
  return X;
}

In [4]:
vector<int> W = {601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110};
vector<int> V = {100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930};
vector<bool> mask = Knapsack(W, V);

int best_val = 0;
int best_weig = 0;

cout << "Los componentes a desplegar en AWS son: " << endl;
for (int i = 0; i < mask.size(); ++i) {
  if (mask[i]) {
    cout << i << ",\n"[i + 1 == mask.size()] << " \n"[i + 1 == mask.size()];
    best_val += V[i];
    best_weig += W[i];
  }
}

cout << "Mejor valor de impacto lograble con 8000MB = " << best_val << " puntos" << endl;
cout << "Memoria RAM usada = " << best_weig << "MB" << endl;

Los componentes a desplegar en AWS son: 
2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46, 49

Mejor valor de impacto lograble con 8000MB = 14121 puntos
Memoria RAM usada = 7775MB


#### Solución óptima del equipo

Después de solucionar el problema usando programación dinámica observamos que el movimiento más inteligente es que de los 50 componentes del sistema que tiene el equipo, se desplieguen en AWS los componentes 2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46 y 49, teniendo en cuenta que los componentes están enumerados desde 0.
Esto proporciona al equipo un valor de impacto total de 14121 puntos usando un total de 7775MB de RAM por parte de estos componentes desplegados.

#### Análisis

## Situación de Set Covering

### Enunciado

Un parque de atracciones atiende a mucha gente a diario, sobre todo en temporadas festivas, y el uso de la maquinaría de las atracciones es muy frecuente, lo que leva a desgastes en las piñonerías, pistones y demás, por lo que el parque debe garantizar que el estado de salud de las atracciones sea correcto en todo momento y así evitar a toda costa un desafortunado accidente.

Para ello el parque de atracciones instaló paquetes de sensores en zonas críticas de cada atracción, cómo lo son los puntos que articulan las máquinas; la evaluación de cierta articulación de una atracción requiere más de un tipo de sensor, es por eso que la instalación no es por sensor individual sino por un paquete de sensores que incluyen un sensor de audio de alta impedancia y de baja impedancia, un sensor hall para campos electromagnéticos, sensor de humedad, 2 termopares para medición de temperatura y 1 foto resistencia para la luminosidad; por fines prácticos a todo un paquete lo llamaremos medidor.

Los medidores ya están instalados en las zonas importantes, sin embargo no están conectados a ninguna red, y por tanto aún no se puede consultar la información desde ningún lado, es por eso que el parque necesita instalar routers alrededor del parque que conecten todos los medidores, los medidores son para IOT por lo que se conectan a los routers inalámbricamente siempre que la distancia sea lo suficientemente pequeña y que no hayan muchos obstáculos en medio para que la señal del medidor llegue al router. Además todos los routers están conectados en una red más grande entre ellos y hacia el internet, sin importar la distancia entre router y router.

El parque aún no sabe qué tantos routers necesita, sin embargo dado el gran tamaño del parque, tiene establecidos 200 posibles lugares en donde podría instalar un router, y en total se deben conectar 50 medidores.

Cada locación de un router tiene definido un subconjunto de los 50 medidores que hay, los cuales conectaría un router en dicho lugar, y cada locación tiene implicitamente un costo de instalación del router, ya que no es lo mismo instalar un router en una habitación cómoda a hacerlo a 30 metros de altura en una posición riesgosa.

El parque quiere seleccionar estratégicamente algunas de esas 200 posibles locaciones, para instalar routers de tal forma que todos los medidores resulten conectados, y adicionalmente esto se debe lograr con el costo total de instalación de routers mínimo, el costo de instalación de un router en cada ubicación está en dólares.

### Imágen del problema

![image.png](park_routers.png)

### Modelamiento matemático del problema

Sea $C$ un listado de 200 costos, en donde $C_i$ es el costo de instalación de un router en la $i-$ésima locación.

Y sea $R$ una matriz de $50$ filas y $200$ columnas que cumple la siguiente condición:

"En la $i-$ésima fila y $j-$ésima columna (celda $R_{i,j}$) hay un $1$ si la $j-$ésima locación de router puede conectar al $i-$ésimo medidor, o en caso contrario la celda tendrá un $0$"

Para entender esta matriz prosigamos al siguiente ejemplo:

| | **Locación 1** | **Locación 2** | **Locación 3**
| :--- | :---: | :---: | :---: |
| **Medidor 1** | 0 | 1 | 1 |
| **Medidor 2** | 1 | 0 | 1 |
| **Medidor 3** | 1 | 0 | 0 |

Acá la locación 1 cubre/conecta los medidores 2 y 3.
La locación 2 cubre solo el Medidor 1.
La locación 3 cubre los medidodres 1 y 2.


#### Variables de decisión

Hay 200 variables binarias, $X_i$ es una variable binaria que indica si se instala un router en la $i-$ésima locación

$1 \leq i \leq 200$

#### Función objetivo

$Min$ $z =$ $\sum_{i=1}^{200}$ $W_i$ $\cdot$ $X_i$ dólares

#### Restricciones

Restricción binaria: $X_i \in \{0, 1\}$ $\forall$ $i \in \mathbb{Z} \cap [1, 200]$

Restricciones de cubrimiento de todos los medidores:

$\forall$ $i \in \mathbb{Z} \cap [1, 200]$, $\sum_{i=1}^{200}$ $R_{i, j}$ $\cdot$ $X_i$ $\geq$ $1$

#### Datos sobre los costos de las locaciones y el cubrimiento de medidores

El modelamiento del problema con los datos de manera completa está en el archivo SetCovering.lp, y en él se encuentra tanto los coeficientes ($C_i$) de cada variable binaria ($X_i$), cómo la información de los cubrimientos de las locaciones, es decir la información de la tabla $R$

Coeficientes ($C_i$) en SetCovering.lp:

![image.png](f_obj_set_covering.png)

Información sobre el cubrimiento de vértices (Tabla $R$):

![image.png](rest_set_covering.png)

Acá la información de cada fila de $R$ está contenida en el listado de variables que se observan en cada (cover_i), básicamente (cover_i) contiene la información de la $i-$ésima fila de $R$, si $x\_j$ aparece en el listado de $cover_i$ quiere decir que $R_{i, j} = 1$, si no aparece, entonces $R_{i, j} = 0$. En pocas palabras cada (cover_i) representa un medidor, y el listado es el listado de qué locaciones cubren a ese medidor.

#### Solución de excel:

La solución completa del problema se encuentra en el archivo set_covering_solution.xlsx adjunto en este repositorio, en dicho archivo de excel se usó solver con el método simplex LP.

![image.png](screenshot_set_covering_excel.png)

#### Análisis de la solución

La mejor decisión por parte del parque de atracciones es instalar 11 routers, de todas las 200 ubicaciones posibles para los routers, las locaciones en las cuáles se deben instalar son las 3, 10, 14, 17, 61, 70, 114, 118, 129, 165, 181. Esto da un total de 91 dólares de costo, el cuál es el mínimo alcanzable.

## Análisis del modelo de redes AS-CAIDA 20071105

A continuación se presenta una imágen de esta red

![image.png](graph_visualization.png)

Aca podemos ver visualmente el grafo que representa la red, a continuación se presenta un análisis de algunas de sus propiedades.

### Cantidad de vértices

Cada nodo en esta red representa una subred operada por una organización independiente, la cantidad de nodos en este grafo es de $26475$.

### Cantidad de aristas

Cada arista en esta red representa una conexión entre dos de las subredes de las organizaciones independientes, la cantidad de aristas en este grafo es de $53381$.

### Densidad

Esta medida en este grafo es menor a 0.000 por lo que este grafo es muy disperso, quiere decir que la relación que hay entre cantidad de nodos y cantidad de aristas, es que hay muchos nodos y pocas aristas.

### Grado máximo

El grado máximo de este grafo es $2628$, quiere decir que en todo el grafo, existe un vértice con $2628$ conexiones a otros vértices, y que no hay otro vértice con más vecinos que este.

### Grado promedio

El grado promedio de este grafo es $4.03$, es decir el promedio de los valores de los grados de todos los vértices en el grafo es $4.03$.

### Cantidad de triángulos

En los grafos muchas veces ocurre que se forman subgrafos en forma de triángulo, es decir subgrafos inducidos los cuáles son grafos completos de 3 vértices ($K_3$), en este grafo la cantidad de triángulos que hay es de $36365$.

### Cantidad máxima de triángulos

Esta métrica hace referencia a la cantidad máxima de triángulos a la que pertenece un solo nodo, en este grafo, existe un nodo que pertenece a 1295 triángulos distintos.

### Cantidad promedio de triángulos

Esta métrica hace referencia a la cantidad promedio de triángulos en los que pertenecen los vértices, es decir por cada vértice obtener la cantidad de triángulos distintos a los que pertenece, y luego promediar todos los valores para todos los vértices, en este grafo la cantidad promedio de triángulos es de $1.2$.

### Local clustering

Esta métrica dice la probabilidad de que dos vértices adyacentes a un nodo, sean adyacentes a su vez entre sí, en este grafo la probabilidad es $0.19$

### assortativity

Esta métrica al ser negativa, ($-0.52$), indica que los nodos muy conectados tienden a conectarse con nodos poco conectados, como por ejemplo en las relaciones cliente/servidor.

### Número cromático

Esta métrica indica la cantidad mínima de colores a pintar cada vértice del grafo tal que ninguna arista conecte dos vértices del mismo color, en este grafo el número cromático es 10, es decir, esto es un grafo k-partito.

### Diámetro

Esta métrica es la distancia máxima entre todos los caminos más cortos entre dos vértices, en este grafo es $8$, quiere decir que hay $2$ vértices cuyo camino más corto es de $8$ y no existe otro par de vértices cuyo _shortest path_ sea mayor a $8$.

### Distancia promedio (Mean distance)

$3.03$ es la distancia promedio en viajar entre dos pares de vértices, lo cuál es bastante bueno para ser una red tan grande.

### Conjunto independiente máximo

El conjunto independiente máximo es un conjunto de vértices tal que ninguna arista del grafo conecta dos vértices de este conjunto, y al mismo tiempo la cardinalidad de este conjunto es maximal, en este grafo la cardinalidad del conjunto independiente máximo es $6136$

### Cubrimiento mínimo de vértices

El cubrimiento mínimo de vértices es un conjunto de vértices tal que para toda arista del grafo, alguno de sus terminales es parte de este conjunto, y a su vez tiene cardinalidad maximal, este conjunto es el complemento del conjunto independiente máximo, y en este grafo la cardinalidad del cubrimiento mínimo de vértices es $20339$

### Conclusion del grafo

Algo demasiado interesante del grafo, y que destaca a simple vista son métricas como la distancia promedio y el diámetro, ya que vemos valores muy bajos para ser un grafo tan grande, esto al ser una topología de red es muy beneficioso ya que el viaje de paquetes es muy eficiente sea cuál sea el par de vértices entre los que tienen que viajar los datos.